# Day E3 — 코호트 정의: 포함·제외, index time, 그리고 불멸 시간 편향

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/E-ehr/day-E3.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/E-ehr/day-E3.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 포함·제외 흐름표

가상의 ICU 입실 5,000건에서 코호트를 좁힙니다.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(3)
n = 5000
stays = pd.DataFrame({
    "subject_id": rng.integers(1, 3800, n),
    "age": rng.integers(15, 95, n),
    "intime": pd.Timestamp("2150-01-01") + pd.to_timedelta(rng.integers(0, 3 * 365 * 24, n), unit="h"),
    "los_hours": rng.gamma(2.0, 30, n).round(1),
    "on_drug_before_icu": rng.random(n) < 0.12,      # ICU 오기 전부터 이미 약을 쓰던 사람
})
flow = [("전체 ICU 입실", len(stays))]
c = stays[stays.age >= 18];                                   flow.append(("18세 이상", len(c)))
c = c.sort_values("intime").groupby("subject_id").head(1);    flow.append(("환자별 첫 ICU 입실만", len(c)))
c = c[c.los_hours >= 24];                                     flow.append(("ICU 24시간 이상 체류", len(c)))
c = c[~c.on_drug_before_icu];                                 flow.append(("입실 전부터 투약 중인 사람 제외", len(c)))
flow = pd.DataFrame(flow, columns=["단계", "남은 수"])
flow["빠진 수"] = (-flow["남은 수"].diff()).fillna(0).astype(int)
print(flow.to_string(index=False))

### 예시 2 — 효과가 0인 약이 사람을 살리는 것처럼 보이는 과정

약이 사망에 전혀 영향을 주지 않도록 만든 가상 세계입니다. 사망 시점은 약과 무관하게 먼저 정해지고, 40%의 환자는 언젠가 약을 받을 후보이지만 **살아 있을 때만** 약을 시작할 수 있습니다.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 20000
death_day = rng.exponential(scale=80, size=n)          # 약과 무관하게 정해진 사망 시점(일)
will_get = rng.random(n) < 0.4                         # 40%는 언젠가 약을 받을 '후보'
start_day = rng.exponential(scale=5, size=n)           # 후보라면 약을 시작했을 시점
treated = will_get & (start_day < np.minimum(death_day, 28))   # 살아 있어야 시작할 수 있다
died28 = death_day < 28
naive = pd.DataFrame({"treated": treated, "died28": died28})
print("[잘못된 분석] 28일 안에 한 번이라도 투약 vs 아님")
print(naive.groupby("treated")["died28"].mean().round(3))

# 고친 분석 A: landmark -- 3일째 살아 있는 사람만, 3일까지의 투약 여부로 나눈다
alive3 = death_day >= 3
lm = pd.DataFrame({"treated_by_day3": (will_get & (start_day < 3))[alive3], "died28": died28[alive3]})
print("\n[landmark 3일] 3일째 생존자, 3일까지 투약 여부로 비교")
print(lm.groupby("treated_by_day3")["died28"].mean().round(3))

# 고친 분석 B: 사람-시간을 노출 전/후로 쪼개 사망률(1,000인-일당) 비교
end = np.minimum(death_day, 28)
unexposed_time = np.where(treated, start_day, end).sum()
exposed_time = np.where(treated, end - start_day, 0).sum()
deaths_unexposed = (died28 & ~treated).sum()
deaths_exposed = (died28 & treated).sum()
print("\n[사람-시간 분할] 1,000인-일당 사망")
print("노출 전 시간:", round(1000 * deaths_unexposed / unexposed_time, 1))
print("노출 후 시간:", round(1000 * deaths_exposed / exposed_time, 1))

### 예시 3 — index time 기준으로 창 자르기

기저 크레아티닌은 index time 전 24시간 안의 마지막 값, 결과는 index time 후 72시간 안의 최댓값으로 정의합니다.

In [ ]:
import pandas as pd

labs = pd.DataFrame({
    "stay_id": [1, 1, 1, 1, 2, 2],
    "charttime": pd.to_datetime(["2150-05-01 03:00", "2150-05-01 09:30", "2150-05-02 01:00",
                                 "2150-05-03 12:00", "2150-07-10 10:00", "2150-07-10 23:00"]),
    "creatinine": [1.1, 1.4, 2.2, 2.9, 0.8, 0.9],
})
index = pd.DataFrame({"stay_id": [1, 2],
                      "index_time": pd.to_datetime(["2150-05-01 08:00", "2150-07-10 12:00"])})
d = labs.merge(index, on="stay_id")
d["hours_from_index"] = (d.charttime - d.index_time).dt.total_seconds() / 3600
baseline = (d[d.hours_from_index.between(-24, 0)]
            .sort_values("charttime").groupby("stay_id").tail(1)[["stay_id", "creatinine"]])
after = d[d.hours_from_index.between(0, 72, inclusive="right")].groupby("stay_id").creatinine.max()
print(d[["stay_id", "charttime", "hours_from_index", "creatinine"]])
print("기저값(index 전 24시간 중 마지막):", baseline.set_index("stay_id").creatinine.to_dict())
print("index 후 72시간 최댓값:", after.to_dict())

### 예시 4 — 코호트 충실도: 알고리즘 vs 차트 검토

가상의 표현형 알고리즘을 의사의 차트 검토 200건과 비교합니다.

In [ ]:
import numpy as np
from sklearn.metrics import cohen_kappa_score

rng = np.random.default_rng(5)
chart = rng.random(200) < 0.3                          # 의사가 차트를 보고 판단한 '진짜' 노출
algo = np.where(chart, rng.random(200) < 0.85, rng.random(200) < 0.08)
tp = (algo & chart).sum(); fp = (algo & ~chart).sum(); fn = (~algo & chart).sum()
print(f"PPV(정밀도) {tp/(tp+fp):.2f}, 민감도(재현율) {tp/(tp+fn):.2f}, "
      f"Cohen's kappa {cohen_kappa_score(chart, algo):.2f}")

## 직접 해보기

1. `notebooks/advanced/E/E3_cohort.ipynb`를 만들고 예시 1–4를 실행합니다. 출력이 위와 같은지 확인합니다.
2. 예시 1에서 "ICU 24시간 이상 체류"와 "환자별 첫 ICU 입실만"의 **순서를 바꿔** 다시 실행하고 최종 인원이 어떻게 달라지는지 적습니다.
3. 예시 2를 그림으로 남깁니다. 같은 셀 아래에 붙여 넣습니다.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

out = Path("../../../figures/advanced/E"); out.mkdir(parents=True, exist_ok=True)
labels = ["naive: never", "naive: ever", "landmark: no", "landmark: yes"]
values = [naive.groupby("treated")["died28"].mean()[False], naive.groupby("treated")["died28"].mean()[True],
          lm.groupby("treated_by_day3")["died28"].mean()[False], lm.groupby("treated_by_day3")["died28"].mean()[True]]
plt.figure(figsize=(6, 3.5))
plt.bar(labels, values, color=["grey", "tab:blue", "grey", "tab:blue"])
plt.ylabel("28-day mortality")
plt.title("Drug with zero true effect")
plt.tight_layout()
plt.savefig(out / "E3_immortal_time.png", dpi=150)
plt.show()